In [1]:
import json
from pyexpat.errors import messages

import openai
from openai.types.chat import ChatCompletionMessage
import requests

client = openai.OpenAI()
api_url = 'https://nomad-movies.nomadcoders.workers.dev'
messages = []


def get_popular_movies():
    response = requests.get(f'{api_url}/movies')
    response.raise_for_status()
    return response.text  # 또는 response.json()


def get_movie_details(id):
    response = requests.get(f'{api_url}/movies/{id}')
    response.raise_for_status()
    return response.text


def get_movie_credits(id):
    response = requests.get(f'{api_url}/movies/{id}/credits')
    response.raise_for_status()
    return response.text


def get_similar_movies(id):
    response = requests.get(f'{api_url}/movies/{id}/similar')
    response.raise_for_status()
    return response.text


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies"
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get a movie's details by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get a movie's cast and crew by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of similar movies",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}],
    tools=TOOLS
)



In [2]:
def process_ai_message(ai_message: ChatCompletionMessage):
    if ai_message.tool_calls:
        for tool_call in ai_message.tool_calls:
            function_name = tool_call.function.name
            function_to_call = FUNCTION_MAP[function_name]
            arguments = tool_call.function.arguments
            print(
                f"Agent: [{function_name} with arguments: {arguments} 호출]")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            if arguments == {}:
                result = function_to_call()
            else:
                result = function_to_call(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result, ensure_ascii=False),
            })

        call_ai()
    else:
        messages.append({"role": "assistant", "content": ai_message.content})
        print(f"AI: {ai_message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )
    ai_message = response.choices[0].message
    assistant_message = {
        "role": "assistant",
        "content": ai_message.content or "",
    }
    if ai_message.tool_calls:
        assistant_message['tool_calls'] = [
            {
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in ai_message.tool_calls
        ]
    messages.append(assistant_message)

    if ai_message.content:
        print(f"AI: {ai_message.content}")
    process_ai_message(response.choices[0].message)


while True:
    message = input("Send a message to the LLM").strip()
    if message == "exit" or message == "quit" or message == "bye":
        print("bye.")
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"You: {messages[-1]['content']}")
        call_ai()



KeyboardInterrupt: Interrupted by user